# Hill Cipher — Polygrafická substituční šifra

**Předmět:** Kryptologie  |  **Implementace:** Python (numpy, sympy)

---


## 1. Historie a motivace

**Hill cipher** je polygrafická substituční šifra, kterou v roce **1929** publikoval americký matematik **Lester S. Hill** v časopise *American Mathematical Monthly*.

### Proč byl vytvořen?
Jednoduché substituční šifry (Caesarova, Vigenèrova) trpí frekvenční analýzou — písmena si zachovávají svou statistickou distribuci. Hill cipher to řeší tím, že šifruje najednou celé bloky písmen pomocí **maticového násobení modulo n**, čímž rozbíjí jednoduchou frekvenční závislost.

### Časová osa
| Rok | Událost |
|-----|---------|
| 1929 | Lester S. Hill publikuje šifru v *American Mathematical Monthly* |
| 1931 | Hill patentuje vylepšenou verzi pro telegrafní přístroje |
| 1950s | Šifra se stává základním výukovým příkladem lineární algebry v kryptografii |
| dnes | Používá se výhradně pro výuku — není bezpečná pro praktické nasazení |

### Tato implementace
Rozšíření pro **českou abecedu** (43 znaků včetně diakritiky, mezery a tečky), volitelný počet šifrovacích kol a libovolná délka klíčové matice.


## 2. Matematické základy

### Abeceda a číselné kódování
Přiřadíme každému znaku abecedy číslo 0 až m−1, kde m je velikost abecedy (zde m = 43).

```text
A→0, Á→1, B→2, C→3, Č→4, D→5, Ď→6, E→7, É→8, Ě→9,
F→10, G→11, H→12, I→13, Í→14, J→15, K→16, L→17, M→18,
N→19, Ň→20, O→21, Ó→22, P→23, Q→24, R→25, Ř→26, S→27,
Š→28, T→29, Ť→30, U→31, Ú→32, Ů→33, V→34, W→35, X→36,
Y→37, Ý→38, Z→39, Ž→40, ' '→41, '.'→42
```

### Šifrování
Text se rozdělí na bloky délky **k** (délka klíčové matice). Každý blok je sloupcový vektor **p** ∈ ℤ_m^k.

```text
c = K · p  (mod m)
```

kde **K** je čtvercová klíčová matice k×k s koeficienty z ℤ_m.

### Dešifrování
```text
p = K⁻¹ · c  (mod m)
```

**K⁻¹** je inverzní matice k **K** modulo m — existuje právě tehdy, když **gcd(det(K), m) = 1**.

### Proč m = 43 a ne 26?
Standardní Hill cipher pracuje s anglickou abecedou (m = 26). Česká abeceda s diakritikou + mezera + tečka = 43 znaků. 43 je prvočíslo, takže inverze modulo 43 existuje pro všechny matice s nenulovým determinantem mod 43 — invertibilita je o něco snazší zaručit než pro m = 26.

### Více kol (Multiple Rounds)
Tato implementace podporuje více kol šifrování. Každé kolo aplikuje jinou klíčovou matici:

```text
c = Kₙ · ... · K₂ · K₁ · p  (mod m)
p = K₁⁻¹ · K₂⁻¹ · ... · Kₙ⁻¹ · c  (mod m)
```


## 3. Pravidla použití

1. **Klíčová matice musí být invertibilní mod 43** — `generate_new_set_of_keys()` to zajišťuje automaticky (sympy `inv_mod`).
2. **Délka textu nemusí být násobkem k** — implementace automaticky doplní padding znakem `Q`; `decypher` ho po dešifrování odřízne.
3. **Znaky mimo abecedu jsou ignorovány** — text je nejprve sanitizován: převeden na velká písmena, neznámé znaky vyhozeny.
4. **Klíč uchovávejte v tajnosti** — sdílejte pouze bezpečným kanálem (Hill cipher sám o sobě výměnu klíčů neřeší).
5. **Nepoužívejte pro produkci** — Hill cipher je lineární a náchylný na útok se známým plaintextem (KPA). Slouží výhradně pro výuku.


## 4. Důsledky porušení protokolu

| Porušení | Důsledek |
|----------|----------|
| Singulární klíčová matice (det ≡ 0 mod 43) | Dešifrování nemožné — K⁻¹ neexistuje |
| Opakované použití stejného klíče s různými plaintexty | KPA útok: útočník sestaví systém rovnic a obnoví K |
| Příliš krátký klíč (k = 1) | Degeneruje na jednoduchý substituční šifrant — okamžitě prolomitelný frekvenční analýzou |
| Útočník zná k plaintext-ciphertext párů bloků | Přímé řešení: K = C · P⁻¹ mod 43 |
| Padding neuložen / špatně odříznut | Dešifrovaný text obsahuje přebytečné Q znaky na konci |


## 5. Kryptoanalýza — Jak by mohl být Hill cipher prolomen?

### Útok se známým plaintextem (Known-Plaintext Attack — KPA)

Pokud útočník zná **k bloků** párů (plaintext, ciphertext), může přímo obnovit klíčovou matici:

```text
C = K · P  (mod m)
→  K = C · P⁻¹  (mod m)
```

kde sloupce P jsou plaintext bloky a sloupce C jsou odpovídající ciphertext bloky. Pokud je P invertibilní mod m, K je plně obnoven po pouhých **k² operacích**.

### Frekvenční analýza
Na rozdíl od monoalfabetických šifer Hill cipher odolává frekvenční analýze jednotlivých znaků. Pro k = 2 jsou digramy (dvojice) statisticky sledovatelné, pro k ≥ 4 je frekvenční analýza prakticky nepoužitelná.

### Útok hrubou silou
Prostor klíčů je obrovský, ale KPA ho dělá ireleventním — k² bloků plaintext-ciphertext páru klíč plně určí.

### Proč Hill cipher nestačí
| Vlastnost | Cesárova šifra | Hill cipher | AES |
|-----------|---------------|-------------|-----|
| Frekvenční analýza | ✗ Triviální | ~ Obtížnější pro k≥4 | ✓ Odolný |
| KPA | ✗ | ✗ k bloků stačí | ✓ Odolný |
| Nelinearita | ✗ | ✗ Čistě lineární | ✓ S-boxy |
| Post-quantum | — | — | ✓ (symetrická) |


In [73]:
# Locally: make sure Hill_Cypher.py is in src/ relative to this notebook.
# In Colab: Files (left panel) → Upload → select Hill_Cypher.py, then adjust the import below.

import sys
sys.path.insert(0, 'src')

from hill_cypher import Hills_cypher

print('Hill Cipher — parametry implementace:')
print(f'  Velikost abecedy (m)     : {len(Hills_cypher.CYPHER_ALPHABET)}')
print(f'  Abeceda                  : {"".join(Hills_cypher.CYPHER_ALPHABET[:10])}... ({len(Hills_cypher.CYPHER_ALPHABET)} znaků)')
print(f'  Výchozí délka klíče (k)  : {Hills_cypher.CA_LEN}  (= velikost abecedy → matice 43×43)')
print()
print('Poznámka: k lze nastavit libovolně při vytvoření Hills_cypher(key_length=k)')
print('Počet kol lze nastavit pomocí Hills_cypher(number_of_rounds=r)')


Hill Cipher — parametry implementace:
  Velikost abecedy (m)     : 43
  Abeceda                  : AÁBCČDĎEÉĚ... (43 znaků)
  Výchozí délka klíče (k)  : 43  (= velikost abecedy → matice 43×43)

Poznámka: k lze nastavit libovolně při vytvoření Hills_cypher(key_length=k)
Počet kol lze nastavit pomocí Hills_cypher(number_of_rounds=r)


## 6. Demo — Generování klíče


In [74]:
import numpy as np
# Vytvoříme šifrovací engine s klíčem velikosti 4×4 (k=4), 1 kolo
engine = Hills_cypher(number_of_rounds=1, key_length=4)
engine.generate_new_set_of_keys()

K   = engine.keys[0]
K_inv = engine.inverse_keys[0]

print(f'Klíčová matice K (4×4, koef. mod 43):')
print(K)
print()
print(f'Inverzní matice K⁻¹ (mod 43):')
print(K_inv)
print()

# Ověřme: K · K⁻¹ ≡ I (mod 43)
product = K.dot(K_inv.astype(int)) % 43
print('Ověření K · K⁻¹ mod 43 (má být jednotková matice):')
print(product % 43)


Klíčová matice K (4×4, koef. mod 43):
[[3 0 0 0]
 [2 2 0 0]
 [3 0 0 3]
 [1 0 3 3]]

Inverzní matice K⁻¹ (mod 43):
[[29 0 0 0]
 [14 22 0 0]
 [5 0 14 29]
 [14 0 29 0]]

Ověření K · K⁻¹ mod 43 (má být jednotková matice):
[[1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]]


## 7. Demo — Šifrování českého textu


In [ ]:
plaintext = (
    'V relačním modelu jsou data uložena v tabulkách, na které má jisté požadavky. '
    'Při splnění požadavků je tabulka označována jako normalizovaná.999999'
)

print('Plaintext:')
print(plaintext)
print()

ciphertext = engine.cypher(plaintext)

print(f'Ciphertext ({len(ciphertext)} znaků):')
print(ciphertext)
print()
print(f'Padding přidán: {engine.padding_length} znaků Q')


Plaintext:
V relačním modelu jsou data uložena v tabulkách, na které má jisté požadavky. Při splnění požadavků je tabulka označována jako normalizovaná.

{'A': 0, 'Á': 1, 'B': 2, 'C': 3, 'Č': 4, 'D': 5, 'Ď': 6, 'E': 7, 'É': 8, 'Ě': 9, 'F': 10, 'G': 11, 'H': 12, 'I': 13, 'Í': 14, 'J': 15, 'K': 16, 'L': 17, 'M': 18, 'N': 19, 'Ň': 20, 'O': 21, 'Ó': 22, 'P': 23, 'Q': 24, 'R': 25, 'Ř': 26, 'S': 27, 'Š': 28, 'T': 29, 'Ť': 30, 'U': 31, 'Ú': 32, 'Ů': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Ý': 38, 'Z': 39, 'Ž': 40, ' ': 41, '.': 42}
{0: 'A', 1: 'Á', 2: 'B', 3: 'C', 4: 'Č', 5: 'D', 6: 'Ď', 7: 'E', 8: 'É', 9: 'Ě', 10: 'F', 11: 'G', 12: 'H', 13: 'I', 14: 'Í', 15: 'J', 16: 'K', 17: 'L', 18: 'M', 19: 'N', 20: 'Ň', 21: 'O', 22: 'Ó', 23: 'P', 24: 'Q', 25: 'R', 26: 'Ř', 27: 'S', 28: 'Š', 29: 'T', 30: 'Ť', 31: 'U', 32: 'Ú', 33: 'Ů', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Ý', 39: 'Z', 40: 'Ž', 41: ' ', 42: '.'}
Ciphertext (144 znaků):
KOYÁÉVÓA.OFNŇĚŠEEJBŠŇMWŤAJYYEF .OĚJÁKOKWĎPGJCÉŽUÍÝNMÁTR.YÚUÝBICGQH

## 8. Demo — Dešifrování správným klíčem


In [76]:
decrypted = engine.decypher(ciphertext)

print('Dešifrovaný text:')
print(decrypted)
print()

# Porovnáme s původním textem po sanitizaci (velká písmena, filtrace neznámých znaků)
sanitized = ''.join(c.upper() for c in plaintext if c.upper() in Hills_cypher.CYPHER_ALPHABET)
print('Shoda s originálem:', decrypted == sanitized)


{'A': 0, 'Á': 1, 'B': 2, 'C': 3, 'Č': 4, 'D': 5, 'Ď': 6, 'E': 7, 'É': 8, 'Ě': 9, 'F': 10, 'G': 11, 'H': 12, 'I': 13, 'Í': 14, 'J': 15, 'K': 16, 'L': 17, 'M': 18, 'N': 19, 'Ň': 20, 'O': 21, 'Ó': 22, 'P': 23, 'Q': 24, 'R': 25, 'Ř': 26, 'S': 27, 'Š': 28, 'T': 29, 'Ť': 30, 'U': 31, 'Ú': 32, 'Ů': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Ý': 38, 'Z': 39, 'Ž': 40, ' ': 41, '.': 42}
{0: 'A', 1: 'Á', 2: 'B', 3: 'C', 4: 'Č', 5: 'D', 6: 'Ď', 7: 'E', 8: 'É', 9: 'Ě', 10: 'F', 11: 'G', 12: 'H', 13: 'I', 14: 'Í', 15: 'J', 16: 'K', 17: 'L', 18: 'M', 19: 'N', 20: 'Ň', 21: 'O', 22: 'Ó', 23: 'P', 24: 'Q', 25: 'R', 26: 'Ř', 27: 'S', 28: 'Š', 29: 'T', 30: 'Ť', 31: 'U', 32: 'Ú', 33: 'Ů', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Ý', 39: 'Z', 40: 'Ž', 41: ' ', 42: '.'}
V RELAČNÍM MODELU JSOU DATA ULOŽENA V TABULKÁCH NA KTERÉ MÁ JISTÉ POŽADAVKY. PŘI SPLNĚNÍ POŽADAVKŮ JE TABULKA OZNAČOVÁNA JAKO NORMALIZOVANÁ.
Dešifrovaný text:
V RELAČNÍM MODELU JSOU DATA ULOŽENA V TABULKÁCH NA KTERÉ MÁ JISTÉ POŽADAVKY. PŘI SPLN

## 9. Demo — Dešifrování s nesprávnou maticí

Vygenerujeme jiný klíč a pokusíme se jím dešifrovat — výsledek bude nesmyslný text.


In [77]:
engine_wrong = Hills_cypher(number_of_rounds=1, key_length=4)
engine_wrong.generate_new_set_of_keys()

# Dešifrujeme ciphertext špatným klíčem
garbage = engine_wrong.decypher(ciphertext)

print('Dešifrování špatným klíčem:')
print(garbage)

print()
print('Správný klíč:')
print(engine.keys[0])
print()
print('Špatný klíč:')
print(engine_wrong.keys[0])


{'A': 0, 'Á': 1, 'B': 2, 'C': 3, 'Č': 4, 'D': 5, 'Ď': 6, 'E': 7, 'É': 8, 'Ě': 9, 'F': 10, 'G': 11, 'H': 12, 'I': 13, 'Í': 14, 'J': 15, 'K': 16, 'L': 17, 'M': 18, 'N': 19, 'Ň': 20, 'O': 21, 'Ó': 22, 'P': 23, 'Q': 24, 'R': 25, 'Ř': 26, 'S': 27, 'Š': 28, 'T': 29, 'Ť': 30, 'U': 31, 'Ú': 32, 'Ů': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Ý': 38, 'Z': 39, 'Ž': 40, ' ': 41, '.': 42}
{0: 'A', 1: 'Á', 2: 'B', 3: 'C', 4: 'Č', 5: 'D', 6: 'Ď', 7: 'E', 8: 'É', 9: 'Ě', 10: 'F', 11: 'G', 12: 'H', 13: 'I', 14: 'Í', 15: 'J', 16: 'K', 17: 'L', 18: 'M', 19: 'N', 20: 'Ň', 21: 'O', 22: 'Ó', 23: 'P', 24: 'Q', 25: 'R', 26: 'Ř', 27: 'S', 28: 'Š', 29: 'T', 30: 'Ť', 31: 'U', 32: 'Ú', 33: 'Ů', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Ý', 39: 'Z', 40: 'Ž', 41: ' ', 42: '.'}
Dešifrování špatným klíčem:
UŮŽUÉ GÚVÚDCGMÍÝUĚÁMZDZŠBUŽÁFN. QZTGŘCÉRŮÉSÚINŇIJÝUGČOVĎDÁYÝWÉPŤOŘÓQÝŤLÓTOŠBSYÝJXĚÁAPCŠĎZHÍSKÚÉNWČRUXĚÚZŮÉSÚKZYŮŮALEĎKÉBXŇOPYWŇĎLŇŇÝXŤĎŽČŽŇNÚFTG

Správný klíč:
[[3 0 0 0]
 [2 2 0 0]
 [3 0 0 3]
 [1 0 3 3]]

Špatný klí

## 10. Demo — Více kol šifrování

Každé kolo aplikuje jinou klíčovou matici. Dešifrování probíhá v opačném pořadí s inverzními maticemi.
Více kol neodstraní lineárnost (součin lineárních zobrazení je lineární), ale ztíží KPA útok — útočník musí znát k bloků pro každé kolo zvlášť nebo pracovat se složenou maticí.


In [78]:
engine3 = Hills_cypher(number_of_rounds=3, key_length=4)
engine3.generate_new_set_of_keys()

print(f'Počet kol: {engine3.rounds}')
print(f'Počet klíčových matic: {len(engine3.keys)}')
print()

ct3 = engine3.cypher(plaintext)
pt3 = engine3.decypher(ct3)

sanitized = ''.join(c.upper() for c in plaintext if c.upper() in Hills_cypher.CYPHER_ALPHABET)

print('Plaintext  :', plaintext[:60], '...')
print('Ciphertext :', ct3[:60], '...')
print('Decrypted  :', pt3[:60], '...')
print()
print('Shoda:', pt3 == sanitized)
print()
print('Poznámka: ciphertext 3 kol se liší od ciphertextu 1 kola:')
print('  1 kolo :', ciphertext[:30])
print('  3 kola :', ct3[:30])


Počet kol: 3
Počet klíčových matic: 3

{'A': 0, 'Á': 1, 'B': 2, 'C': 3, 'Č': 4, 'D': 5, 'Ď': 6, 'E': 7, 'É': 8, 'Ě': 9, 'F': 10, 'G': 11, 'H': 12, 'I': 13, 'Í': 14, 'J': 15, 'K': 16, 'L': 17, 'M': 18, 'N': 19, 'Ň': 20, 'O': 21, 'Ó': 22, 'P': 23, 'Q': 24, 'R': 25, 'Ř': 26, 'S': 27, 'Š': 28, 'T': 29, 'Ť': 30, 'U': 31, 'Ú': 32, 'Ů': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Ý': 38, 'Z': 39, 'Ž': 40, ' ': 41, '.': 42}
{0: 'A', 1: 'Á', 2: 'B', 3: 'C', 4: 'Č', 5: 'D', 6: 'Ď', 7: 'E', 8: 'É', 9: 'Ě', 10: 'F', 11: 'G', 12: 'H', 13: 'I', 14: 'Í', 15: 'J', 16: 'K', 17: 'L', 18: 'M', 19: 'N', 20: 'Ň', 21: 'O', 22: 'Ó', 23: 'P', 24: 'Q', 25: 'R', 26: 'Ř', 27: 'S', 28: 'Š', 29: 'T', 30: 'Ť', 31: 'U', 32: 'Ú', 33: 'Ů', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Ý', 39: 'Z', 40: 'Ž', 41: ' ', 42: '.'}
{'A': 0, 'Á': 1, 'B': 2, 'C': 3, 'Č': 4, 'D': 5, 'Ď': 6, 'E': 7, 'É': 8, 'Ě': 9, 'F': 10, 'G': 11, 'H': 12, 'I': 13, 'Í': 14, 'J': 15, 'K': 16, 'L': 17, 'M': 18, 'N': 19, 'Ň': 20, 'O': 21, 'Ó': 22, 'P': 23